In [1]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & GPU SELECTION (Inspired by your notebook)
# ==============================================================================
import os
import shutil
import subprocess
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import glob
from tqdm import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

def get_free_gpu():
    try:
        result = subprocess.check_output(
            "nvidia-smi --query-gpu=memory.free --format=csv,nounits,noheader", shell=True)
        memory_free = [int(x) for x in result.decode().strip().split("\n")]
        return memory_free.index(max(memory_free))
    except:
        return 0

os.environ["CUDA_VISIBLE_DEVICES"] = str(get_free_gpu())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Selected GPU: {device}")

✅ Selected GPU: cuda


In [2]:
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
if os.path.exists("kaggle.json"):
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("Kaggle credentials configured!")

Kaggle credentials configured!


In [3]:
os.system("rm -rf ./btp_raw_data")
os.system("rm -rf ./btp_3class_dataset")

0

In [ ]:
# ==========================================
# CELL 2: DATASET PREPARATION
# ==========================================
import os
import shutil
import glob
from sklearn.model_selection import train_test_split

def prepare_dataset():
    print("🧹 1. Cleaning up old directories...")
    os.system("rm -rf /content/btp_raw_data")
    os.system("rm -rf /content/btp_3class_dataset")

    BASE_RAW = "/content/btp_raw_data"
    UNIFIED_PATH = "/content/btp_3class_dataset"
    os.makedirs(BASE_RAW, exist_ok=True)

    # Kaggle datasets
    datasets_to_get = {
        "pneumonia": "paultimothymooney/chest-xray-pneumonia",
        "covid": "tawsifurrahman/covid19-radiography-database"
    }

    print("📊 2. Downloading reliable datasets via Kaggle API...")
    for folder, slug in datasets_to_get.items():
        dest = f"{BASE_RAW}/{folder}"
        if not os.path.exists(dest):
            print(f"📥 Downloading {folder} dataset...")
            os.system(f"kaggle datasets download -d {slug} -p {dest}")
            os.system(f"unzip -q {dest}/*.zip -d {dest}")
            os.system(f"rm {dest}/*.zip")
            print(f"✅ {folder} extracted.")

    print("\n🚀 3. Building balanced 3-Class dataset...")
    for split in ["train", "val", "test"]:
        for cls in ["Normal", "COVID", "Pneumonia"]:
            os.makedirs(os.path.join(UNIFIED_PATH, split, cls), exist_ok=True)

    pneumonia_raw_dir = f"{BASE_RAW}/pneumonia/chest_xray"
    sample_size = 1500
    
    normal_imgs = glob.glob(f"{pneumonia_raw_dir}/**/NORMAL/*.jpeg", recursive=True)[:sample_size]
    pneumonia_imgs = glob.glob(f"{pneumonia_raw_dir}/**/PNEUMONIA/*.jpeg", recursive=True)[:sample_size]
    covid_imgs = glob.glob(f"{BASE_RAW}/covid/**/COVID/images/*.png", recursive=True)[:sample_size]

    def split_and_copy(files, class_name):
        if not files: return
        train, temp = train_test_split(files, test_size=0.3, random_state=42)
        val, test = train_test_split(temp, test_size=0.5, random_state=42)
        
        for subset, s_name in zip([train, val, test], ["train", "val", "test"]):
            for f in subset:
                shutil.copy(f, os.path.join(UNIFIED_PATH, s_name, class_name))

    print("Copying images to train/val/test folders...")
    split_and_copy(normal_imgs, 'Normal')
    split_and_copy(pneumonia_imgs, 'Pneumonia')
    split_and_copy(covid_imgs, 'COVID')
    
    print(f"\n🎉 SUCCESS! Dataset organized at: {os.path.abspath(UNIFIED_PATH)}")

prepare_dataset()

In [8]:
# ==============================================================================
# 3. DATASET ORGANIZATION (Normal, COVID, Pneumonia)
# ==============================================================================
def organize_data(sample_size=1500):
    for split in ["train", "val", "test"]:
        for cls in ["Normal", "COVID", "Pneumonia"]:
            os.makedirs(os.path.join(UNIFIED_PATH, split, cls), exist_ok=True)

    df = pd.read_csv(f"{nih_path}/{metadata_file}", encoding='latin1')
    available_imgs = [os.path.basename(x) for x in glob.glob(f"{nih_path}/images/*.png")]
    df_filtered = df[df['Image Index'].isin(available_imgs)]
    
    normal_df = df_filtered[df_filtered['Finding Labels'] == 'No Finding'].sample(sample_size, random_state=42)
    pneumonia_df = df_filtered[df_filtered['Finding Labels'].str.contains('Pneumonia')].sample(sample_size, random_state=42)
    covid_img_paths = glob.glob(f"{covid_path}/**/COVID/images/*.png", recursive=True)[:sample_size]

    def split_and_copy(files, class_name, is_nih=False):
        paths = [os.path.join(nih_path, "images", f) for f in files] if is_nih else files
        train, temp = train_test_split(paths, test_size=0.3, random_state=42)
        val, test = train_test_split(temp, test_size=0.5, random_state=42)
        for subset, s_name in zip([train, val, test], ["train", "val", "test"]):
            for f in subset:
                shutil.copy(f, os.path.join(UNIFIED_PATH, s_name, class_name))

    print("Organizing 3-class splits...")
    split_and_copy(normal_df['Image Index'].tolist(), 'Normal', is_nih=True)
    split_and_copy(pneumonia_df['Image Index'].tolist(), 'Pneumonia', is_nih=True)
    split_and_copy(covid_img_paths, 'COVID', is_nih=False)

if not os.path.exists(os.path.join(UNIFIED_PATH, "train")):
    organize_data()

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using Device: {device}")

UNIFIED_PATH = "./btp_3class_dataset"
IMG_SIZE, BATCH = 224, 32

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loaders = {x: DataLoader(datasets.ImageFolder(f"{UNIFIED_PATH}/{x}", transform), 
                         batch_size=BATCH, shuffle=(x=='train')) for x in ['train', 'val', 'test']}

model_list = ["resnet50", "efficientnet_b0", "densenet121", "vgg16", "vit_base_patch16_224"]

def train_base_models(epochs=10):
    trained_models = {}
    for name in model_list:
        print(f"\n🚀 Training {name}...")
        model = timm.create_model(name, pretrained=True, num_classes=3).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        best_acc = 0.0

        for epoch in range(epochs):
            model.train()
            for imgs, labels in tqdm(loaders['train'], desc=f"Epoch {epoch+1}/{epochs}"):
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()
            
            # Validation Checkpoint
            model.eval()
            preds, targets = [], []
            with torch.no_grad():
                for imgs, labels in loaders['val']:
                    out = model(imgs.to(device))
                    preds.extend(torch.argmax(out, 1).cpu().numpy())
                    targets.extend(labels.numpy())
            
            acc = accuracy_score(targets, preds)
            if acc > best_acc:
                best_acc = acc
                torch.save(model.state_dict(), f"best_{name}_3class.pth")
                print(f"   ⭐ Saved best model! Val Acc: {acc:.4f}")
        
        # Load best weights into memory for the ensemble/robustness steps
        model.load_state_dict(torch.load(f"best_{name}_3class.pth"))
        trained_models[name] = model
        
    return trained_models

# Run Training
models_dict = train_base_models(epochs=10)

✅ Using Device: cuda

🚀 Training resnet50...


Epoch 1/10: 100%|███████████████████████████████████████████████████████████████████████| 99/99 [03:46<00:00,  2.28s/it]


   ⭐ Saved best model! Val Acc: 0.8993


Epoch 2/10: 100%|███████████████████████████████████████████████████████████████████████| 99/99 [03:42<00:00,  2.24s/it]


   ⭐ Saved best model! Val Acc: 0.9037


Epoch 3/10: 100%|███████████████████████████████████████████████████████████████████████| 99/99 [01:59<00:00,  1.20s/it]


... (40 outputs left out)

   ⭐ Saved best model! Val Acc: 0.9748


Epoch 4/10: 100%|███████████████████████████████████████████████████████████████████████| 99/99 [01:02<00:00,  1.58it/s]


   ⭐ Saved best model! Val Acc: 0.9763


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████████| 99/99 [02:26<00:00,  1.48s/it]


In [9]:
# ==============================================================================
# 5. OPTUNA ENSEMBLE WEIGHTING (On Clean Validation Data)
# ==============================================================================
import optuna
from sklearn.metrics import roc_auc_score
print("\n" + "="*50 + "\n🎯 CALCULATING OPTIMAL ENSEMBLE WEIGHTS\n" + "="*50)

val_preds_all, val_targets_list = [], []

for name in model_list:
    model = models_dict[name]
    model.eval()
    model_probs = []
    
    with torch.no_grad():
        for imgs, labels in loaders['val']:
            imgs = imgs.to(device)
            out = model(imgs)
            probs = torch.softmax(out, dim=1) 
            model_probs.extend(probs.cpu().numpy())
            if name == model_list[0]: # Only need to collect targets once
                val_targets_list.extend(labels.numpy())
                
    val_preds_all.append(np.array(model_probs))

val_targets = np.array(val_targets_list)

# Optuna Objective: Maximize Multi-class AUC
def objective(trial):
    w = [trial.suggest_float(f"w{i}", 0, 1) for i in range(len(model_list))]
    w = np.array(w) / sum(w) # Normalize weights
    
    ensemble_pred = np.zeros_like(val_preds_all[0])
    for i in range(len(model_list)):
        ensemble_pred += w[i] * val_preds_all[i]
    
    return roc_auc_score(val_targets, ensemble_pred, average="macro", multi_class='ovr')

optuna.logging.set_verbosity(optuna.logging.WARNING) # Keep output clean
study = optuna.create_study(direction="maximize")
print("⏳ Running Optuna trials to find optimal weights...")
study.optimize(objective, n_trials=100)

best_w = np.array([study.best_params[f"w{i}"] for i in range(len(model_list))])
best_w = best_w / best_w.sum()

print("👑 BEST ENSEMBLE WEIGHTS FOUND:")
for i, name in enumerate(model_list):
    print(f" - {name:<20} : {best_w[i]:.4f} ({best_w[i]*100:.1f}%)")


🎯 CALCULATING OPTIMAL ENSEMBLE WEIGHTS
⏳ Running Optuna trials to find optimal weights...
👑 BEST ENSEMBLE WEIGHTS FOUND:
 - resnet50             : 0.0494 (4.9%)
 - efficientnet_b0      : 0.0866 (8.7%)
 - densenet121          : 0.3374 (33.7%)
 - vgg16                : 0.2034 (20.3%)
 - vit_base_patch16_224 : 0.3232 (32.3%)


In [10]:
# ==============================================================================
# 6. CLINICAL NOISE & ROBUSTNESS ANALYSIS
# ==============================================================================
class ClinicalNoise:
    @staticmethod
    def apply_poisson(img, sigma):
        peak = 1.0 / (sigma + 1e-6)
        return torch.clamp(torch.poisson(img * peak) / peak, 0, 1)

    @staticmethod
    def apply_speckle(img, sigma):
        return img + img * torch.randn_like(img) * sigma

    @staticmethod
    def apply_salt_pepper(img, sigma):
        noisy = img.clone()
        mask = torch.rand_like(img)
        noisy[mask < (sigma/2)] = 0
        noisy[mask > (1 - sigma/2)] = 1
        return noisy

    @staticmethod
    def apply_gaussian(img, sigma):
        return img + sigma * torch.randn_like(img)

noise_types = ["Poisson", "Speckle", "Salt&Pepper", "Gaussian"]
levels = [0.01, 0.05, 0.1, 0.2]

plot_models = model_list + ["Optimized_Ensemble"]
robustness_results = {name: {nt: [] for nt in noise_types} for name in plot_models}

print("\n" + "="*50 + "\n🔬 RUNNING NOISE ROBUSTNESS ANALYSIS\n" + "="*50)
for nt in noise_types:
    print(f"Evaluating degradation under {nt} noise...")
    for lvl in levels:
        test_preds = {name: [] for name in model_list}
        test_targets_list = []
        
        with torch.no_grad():
            for imgs, labels in loaders['test']:
                imgs = imgs.to(device)
                
                # Inject Clinical Noise
                if nt == "Poisson": noisy = ClinicalNoise.apply_poisson(imgs, lvl)
                elif nt == "Speckle": noisy = ClinicalNoise.apply_speckle(imgs, lvl)
                elif nt == "Salt&Pepper": noisy = ClinicalNoise.apply_salt_pepper(imgs, lvl)
                else: noisy = ClinicalNoise.apply_gaussian(imgs, lvl)
                
                test_targets_list.extend(labels.numpy())
                
                for name in model_list:
                    model = models_dict[name]
                    model.eval()
                    out = model(noisy)
                    probs = torch.softmax(out, dim=1)
                    test_preds[name].extend(probs.cpu().numpy())
        
        # Log Base Model Accuracies
        for name in model_list:
            preds_class = np.argmax(np.array(test_preds[name]), axis=1)
            acc = accuracy_score(test_targets_list, preds_class)
            robustness_results[name][nt].append(acc)
            
        # Log Optimized Ensemble Accuracy
        ensemble_probs = np.zeros_like(test_preds[model_list[0]])
        for i, name in enumerate(model_list):
            ensemble_probs += best_w[i] * np.array(test_preds[name])
        
        ensemble_class = np.argmax(ensemble_probs, axis=1)
        ensemble_acc = accuracy_score(test_targets_list, ensemble_class)
        robustness_results["Optimized_Ensemble"][nt].append(ensemble_acc)